# PlantSeg -- Array 2nd-Revision Notebook B: Multi-Seed Diagnostics (seeds 42, 123, 7)

**Pure inference. No training.** This notebook reuses the checkpoints already produced by
the completed multi-seed retraining pass (seed 123 and seed 7, trained specifically for the
round-1 variance/robustness response) alongside the original seed-42 checkpoints. It answers
the round-2 reviewers' concern that single-seed point estimates (model selection, the
confidence-flag validation, faithfulness numbers) might not be robust to the training seed.

## What this notebook answers

1. **Section 1 -- Validation-split model re-ranking across 3 seeds.** Repeats Stage 6
   Section 2's "select the best model on the validation set, not the test set" check, but
   now at all 3 seeds (42, 123, 7) for both the 8 classifiers and the 6 segmenters, so the
   paper can report whether the same architecture wins consistently rather than a single
   seed's re-ranking.
2. **Section 2 -- Validation-split confidence-flag check across 3 seeds.** Repeats Stage 5's
   flag-validation tests (Fisher's exact: flag vs. correctness; Mann-Whitney U: flag vs.
   severity error) on the **validation** split, using each seed's own ConvNeXtV2-Tiny +
   DeepLabV3+/EfficientNet-B3 checkpoints, to show the deployed confidence flag's validity is
   not an artifact of one particular training run.
3. **Section 3 -- Consistent 3-seed segmentation / severity / attribution recompute.** For
   all 6 segmenters at all 3 seeds: test-set fg-IoU/Dice (Stage 2 style), severity CCC/MAE
   against `Mask ratio` (Stage 3 style), and CAM-GT faithfulness on the same 300-image
   stratified subset (Stage 4 style) -- combined into mean +/- std-across-seeds tables, so
   the paper can report variance bands instead of single-seed point estimates.

## Required Kaggle inputs (attach as datasets)
- The PlantSeg image + annotation dataset.
- The 8 original SEED=42 classifier checkpoints + the 6 original SEED=42 segmenter
  checkpoints (same datasets Notebook A and Stage 4/5/6 use).
- The seed-123 and seed-7 **classifier** checkpoints (`results.zip` / `results (1).zip` from
  the completed multi-seed retraining -- each contains all 8 classifiers for that seed as
  `s1_results_seed{seed}/{Model}_seed{seed}_best.pt`).
- The seed-123 and seed-7 **segmenter** checkpoints from the completed multi-seed retraining
  lanes (`Lane_1`..`Lane_6` zips) -- attach all of them as one or more datasets; the
  checkpoint finder below searches recursively under `/kaggle/input` so lane/folder naming
  does not matter.

## A note on the seed-123/seed-7 segmenter checkpoint files
Those `.zip` files are, per inspection, **already PyTorch's own zip-serialized checkpoint
format** saved with a `.zip` extension rather than `.pt` -- `torch.load()` detects the format
from file content, not the extension, so in most cases they load with a plain
`torch.load(path)` even though the name ends in `.zip`. The one failure mode is if Kaggle's
dataset uploader **auto-extracts** a `.zip` into a loose folder when the dataset is created
(common behavior) -- in that case the checkpoint's internal files (`data.pkl`, `data/0`,
`data/1`, ...) end up as a plain directory, which `torch.load` cannot open directly. The
`find_and_load_checkpoint()` helper below handles both cases: it first tries a direct
`torch.load` on any matching file, and if that fails (or only a matching *directory* is
found), it re-zips that directory's contents in memory and loads the result. If it still
fails for a specific checkpoint, the cell prints which file/seed/model failed so that can be
reported back and fixed by hand (e.g. re-uploading that one lane as a single-file dataset).

## Estimated runtime
All three sections are inference-only over the validation set (846 images) and/or the
300-image XAI subset, repeated 3x for the 3 seeds. Expect roughly **3-5 hours total**,
comfortably inside the 12-hour session limit. Each section writes its own CSV before moving
on to the next, so a partial run still leaves usable output.

## Setup

In [1]:
!pip install -q timm segmentation-models-pytorch grad-cam transformers
import os, json, glob, random, warnings, io, zipfile
from pathlib import Path
from collections import defaultdict
import numpy as np, pandas as pd
from PIL import Image, ImageDraw
import torch, torch.nn as nn, torch.nn.functional as F
import torchvision
import torchvision.transforms as T
import timm
import segmentation_models_pytorch as smp
from transformers import SegformerForSemanticSegmentation
from pytorch_grad_cam import GradCAM
from sklearn.metrics import accuracy_score, f1_score, matthews_corrcoef
from scipy import stats
from scipy.stats import fisher_exact, mannwhitneyu
warnings.filterwarnings("ignore")

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.5/44.5 kB 2.2 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 154.8/154.8 kB 9.0 MB/s eta 0:00:00
Device: cuda


## Config

In [2]:
# ============ CONFIG ============
def find_data_root():
    for cand in [Path("/kaggle/input/datasets/alenmaruf/plantseg/plantseg"),
                 Path("/kaggle/input/plantseg"), Path("plantseg"), Path(".")]:
        if (cand/"Metadata.csv").exists(): return cand
    hits = glob.glob("/kaggle/input/**/Metadata.csv", recursive=True)
    assert hits, "Metadata.csv not found anywhere under /kaggle/input -- attach the PlantSeg dataset."
    return Path(hits[0]).parent

DATA_ROOT = find_data_root()
print("DATA_ROOT:", DATA_ROOT)

OUTDIR = Path("/kaggle/working/r2_notebookB_results"); OUTDIR.mkdir(parents=True, exist_ok=True)

IMG_SIZE_CLS = 384; IMG_SIZE_CLS_TF = 224; IMG_SIZE_SEG = 512
MEAN = [0.485, 0.456, 0.406]; STD = [0.229, 0.224, 0.225]

meta = pd.read_csv(DATA_ROOT/"Metadata.csv", encoding="utf-8-sig")
meta.columns = [c.strip() for c in meta.columns]
meta["split"] = meta["Split"].map({"Training":"train","Validation":"val","Test":"test"})
meta["filepath"] = meta.apply(lambda r: DATA_ROOT/"images"/r["split"]/r["Name"], axis=1)
meta["maskpath"] = meta.apply(lambda r: DATA_ROOT/"annotations"/r["split"]/r["Label file"], axis=1) \
    if "Label file" in meta.columns else None
meta["label"] = meta["Disease"]
CLASS_NAMES = sorted(meta["label"].unique().tolist())
NUM_CLASSES = len(CLASS_NAMES)
label2idx = {c: i for i, c in enumerate(CLASS_NAMES)}

train_df = meta[meta.split=="train"].reset_index(drop=True)
val_df   = meta[meta.split=="val"].reset_index(drop=True)
test_df  = meta[meta.split=="test"].reset_index(drop=True)

if meta["maskpath"].isna().any() or not meta["maskpath"].apply(lambda p: p is not None and Path(p).exists()).all():
    MASK = Path("/kaggle/working/masks")
    def load_anns(split):
        d = json.load(open(DATA_ROOT/f"annotation_{split}.json")); by = defaultdict(list)
        for a in d["annotations"]: by[a["image_id"]].append(a)
        fn = {}
        for im in d["images"]:
            polys = [seg for a in by.get(im["id"], []) for seg in a["segmentation"] if len(seg) >= 6]
            fn[im["file_name"]] = (im["width"], im["height"], polys)
        return fn
    def rasterize(w, h, polys):
        m = Image.new("L", (w, h), 0); dr = ImageDraw.Draw(m)
        for seg in polys: dr.polygon(list(zip(seg[0::2], seg[1::2])), fill=1)
        return m
    mp_by_name = {}
    for split in ["train", "val", "test"]:
        fn = load_anns(split); od = MASK/split; od.mkdir(parents=True, exist_ok=True)
        for r in meta[meta.split == split].itertuples():
            w, h, polys = fn.get(r.Name, (None, None, []))
            mp = od/(Path(r.Name).stem + ".png")
            if not mp.exists():
                if w is None:
                    with Image.open(r.filepath) as i0: w, h = i0.size
                rasterize(w, h, polys).save(mp)
            mp_by_name[r.Name] = mp
    meta["maskpath"] = meta["Name"].map(mp_by_name)
    train_df = meta[meta.split=="train"].reset_index(drop=True)
    val_df   = meta[meta.split=="val"].reset_index(drop=True)
    test_df  = meta[meta.split=="test"].reset_index(drop=True)
    print("Rasterized masks from annotation_*.json.")

print(f"classes={NUM_CLASSES}  train={len(train_df)}  val={len(val_df)}  test={len(test_df)}")

CLS_NAMES_ALL = ["MobileNetV3Large","EfficientNetB0","ResNet50","EfficientNetV2S",
                 "ConvNeXtV2Tiny","ViTBase16","SwinTiny","GatedFusionLight"]
SEG_NAMES_ALL = ["UNet_resnet34","UNetPP_resnet50","FPN_resnet50",
                 "DeepLabV3Plus_resnet50","DeepLabV3Plus_efficientnet-b3","SegFormer_b2"]
SEEDS = [42, 123, 7]

def cls_ckpt_candidates(name, seed):
    return [f"{name}_best.pt"] if seed == 42 else [f"{name}_seed{seed}_best.pt"]

def seg_ckpt_candidates(name, seed):
    return [f"{name}.pt"] if seed == 42 else [f"{name}_seed{seed}.pt", f"{name}_seed{seed}.zip"]

print("Config ready.")

DATA_ROOT: /kaggle/input/datasets/alenmaruf/plantseg/plantseg
classes=115  train=5367  val=846  test=1561
Config ready.


## Robust checkpoint finder (handles both the plain-file and Kaggle-auto-extracted-zip cases)

In [3]:
# ============ ROBUST CHECKPOINT FINDER ============
def _rezip_dir_as_pt(dirpath: Path):
    # Re-zip an already-extracted PyTorch zip-serialized checkpoint folder back into an
    # in-memory buffer that torch.load can read -- needed only if Kaggle auto-extracted the
    # checkpoint's .zip on dataset creation instead of keeping it as a single file.
    root_name = dirpath.name
    buf = io.BytesIO()
    with zipfile.ZipFile(buf, "w", zipfile.ZIP_STORED) as zf:
        for f in dirpath.rglob("*"):
            if f.is_file():
                zf.write(f, arcname=f"{root_name}/{f.relative_to(dirpath).as_posix()}")
    buf.seek(0)
    return buf

_ckpt_cache = {}

def find_and_load_checkpoint(filename_candidates, search_root="/kaggle/input"):
    # filename_candidates: exact filenames to search for, in priority order. Returns the
    # loaded raw object (torch.load result, usually a state_dict) or raises FileNotFoundError.
    key = tuple(filename_candidates)
    if key in _ckpt_cache:
        return _ckpt_cache[key]
    for fname in filename_candidates:
        hits = glob.glob(f"{search_root}/**/{fname}", recursive=True)
        for h in hits:
            hp = Path(h)
            try:
                state = torch.load(hp, map_location="cpu")
                print(f"    [find_and_load] loaded {hp}")
                _ckpt_cache[key] = state; return state
            except Exception as e:
                print(f"    [find_and_load] {hp} exists but torch.load failed directly ({type(e).__name__}: {e});"
                      f" checking for an auto-extracted folder instead...")
        stem = Path(fname).stem
        dir_hits = [d for d in glob.glob(f"{search_root}/**/{stem}", recursive=True) if Path(d).is_dir()]
        for d in dir_hits:
            dp = Path(d)
            if (dp / "data.pkl").exists():
                print(f"    [find_and_load] found extracted checkpoint folder {dp}; re-zipping in memory")
                state = torch.load(_rezip_dir_as_pt(dp), map_location="cpu")
                _ckpt_cache[key] = state; return state
            inner = dp / stem
            if inner.is_dir() and (inner / "data.pkl").exists():
                print(f"    [find_and_load] found nested extracted checkpoint folder {inner}; re-zipping in memory")
                state = torch.load(_rezip_dir_as_pt(inner), map_location="cpu")
                _ckpt_cache[key] = state; return state
    raise FileNotFoundError(f"None of {filename_candidates} found (as file or extracted folder) under {search_root}")

def apply_state(model, raw):
    if isinstance(raw, dict):
        for key in ("state_dict", "model_state_dict", "model", "best_state"):
            if key in raw and isinstance(raw[key], dict):
                raw = raw[key]; break
    raw = {k.replace("module.", ""): v for k, v in raw.items()}
    missing, unexpected = model.load_state_dict(raw, strict=False)
    if missing or unexpected:
        print(f"    [apply_state] missing={len(missing)} unexpected={len(unexpected)}")
        total_model_keys = len(model.state_dict())
        if total_model_keys and len(missing) >= total_model_keys * 0.5:
            print(f"    [apply_state] sample checkpoint keys: {list(raw.keys())[:3]}")
            print(f"    [apply_state] sample expected keys  : {list(model.state_dict().keys())[:3]}")
            raise RuntimeError(f">=50% of keys unmatched (missing={len(missing)}, unexpected={len(unexpected)}) "
                                f"-- checkpoint did not load; refusing to report numbers from random weights")
    return model

# Quick inventory check -- prints what was found for each (model, seed) before any heavy
# computation runs, so a missing dataset attachment is caught immediately.
print("Checkpoint inventory check:")
for seed in SEEDS:
    for name in CLS_NAMES_ALL:
        cands = cls_ckpt_candidates(name, seed)
        found = any(glob.glob(f"/kaggle/input/**/{c}", recursive=True) or
                    [d for d in glob.glob(f"/kaggle/input/**/{Path(c).stem}", recursive=True) if Path(d).is_dir()]
                    for c in cands)
        print(f"  seed{seed:>3} classifier {name:<18} {'OK' if found else 'MISSING'}")
    for name in SEG_NAMES_ALL:
        cands = seg_ckpt_candidates(name, seed)
        found = any(glob.glob(f"/kaggle/input/**/{c}", recursive=True) or
                    [d for d in glob.glob(f"/kaggle/input/**/{Path(c).stem}", recursive=True) if Path(d).is_dir()]
                    for c in cands)
        print(f"  seed{seed:>3} segmenter {name:<30} {'OK' if found else 'MISSING'}")

Checkpoint inventory check:
  seed 42 classifier MobileNetV3Large   OK
  seed 42 classifier EfficientNetB0     OK
  seed 42 classifier ResNet50           OK
  seed 42 classifier EfficientNetV2S    OK
  seed 42 classifier ConvNeXtV2Tiny     OK
  seed 42 classifier ViTBase16          OK
  seed 42 classifier SwinTiny           OK
  seed 42 classifier GatedFusionLight   OK
  seed 42 segmenter UNet_resnet34                  OK
  seed 42 segmenter UNetPP_resnet50                OK
  seed 42 segmenter FPN_resnet50                   OK
  seed 42 segmenter DeepLabV3Plus_resnet50         OK
  seed 42 segmenter DeepLabV3Plus_efficientnet-b3  OK
  seed 42 segmenter SegFormer_b2                   OK
  seed123 classifier MobileNetV3Large   OK
  seed123 classifier EfficientNetB0     OK
  seed123 classifier ResNet50           OK
  seed123 classifier EfficientNetV2S    OK
  seed123 classifier ConvNeXtV2Tiny     OK
  seed123 classifier ViTBase16          OK
  seed123 classifier SwinTiny           OK
  s

## Model definitions (shared across sections)

In [4]:
# ============ MODEL DEFINITIONS ============
def build_classifier(num_classes):
    return timm.create_model("convnextv2_tiny", pretrained=False, num_classes=num_classes)

def build_backbone(name, num_classes):
    if name == "MobileNetV3Large":
        m = torchvision.models.mobilenet_v3_large(weights=None)
        m.classifier[-1] = nn.Linear(m.classifier[-1].in_features, num_classes); return m, IMG_SIZE_CLS
    if name == "EfficientNetB0":
        return timm.create_model("efficientnet_b0", pretrained=False, num_classes=num_classes), IMG_SIZE_CLS
    if name == "ResNet50":
        m = torchvision.models.resnet50(weights=None)
        m.fc = nn.Linear(m.fc.in_features, num_classes); return m, IMG_SIZE_CLS
    if name == "EfficientNetV2S":
        return timm.create_model("tf_efficientnetv2_s", pretrained=False, num_classes=num_classes), IMG_SIZE_CLS
    if name == "ConvNeXtV2Tiny":
        return timm.create_model("convnextv2_tiny", pretrained=False, num_classes=num_classes), IMG_SIZE_CLS
    if name == "ViTBase16":
        return timm.create_model("vit_base_patch16_224", pretrained=False, num_classes=num_classes), IMG_SIZE_CLS_TF
    if name == "SwinTiny":
        return timm.create_model("swin_tiny_patch4_window7_224", pretrained=False, num_classes=num_classes), IMG_SIZE_CLS_TF
    if name == "GatedFusionLight":
        class _Gated(nn.Module):
            def __init__(self, ba, da, bb, db, nc, gh=128, hh=256, dr=0.3):
                super().__init__(); self.branch_a=ba; self.branch_b=bb; fused=da+db
                self.gate = nn.Sequential(nn.Linear(fused,gh), nn.ReLU(), nn.Linear(gh,2), nn.Softmax(dim=1))
                self.classifier = nn.Sequential(nn.Linear(fused,hh), nn.ReLU(), nn.Dropout(dr), nn.Linear(hh,nc))
            def forward(self, x):
                fa, fb = self.branch_a(x), self.branch_b(x)
                g = self.gate(torch.cat([fa,fb], 1))
                return self.classifier(torch.cat([fa*g[:,0:1], fb*g[:,1:2]], 1))
        mb = torchvision.models.mobilenet_v3_large(weights=None)
        da = mb.classifier[0].in_features; mb.classifier = nn.Identity()
        b = timm.create_model("efficientnet_b0", pretrained=False, num_classes=0, global_pool="avg")
        return _Gated(mb, da, b, b.num_features, num_classes), IMG_SIZE_CLS_TF
    raise ValueError(name)

SMP_SPECS = {
    "UNet_resnet34":                 (smp.Unet,          "resnet34"),
    "UNetPP_resnet50":               (smp.UnetPlusPlus,  "resnet50"),
    "DeepLabV3Plus_resnet50":        (smp.DeepLabV3Plus, "resnet50"),
    "DeepLabV3Plus_efficientnet-b3": (smp.DeepLabV3Plus, "efficientnet-b3"),
    "FPN_resnet50":                  (smp.FPN,           "resnet50"),
}
SEGFORMER_HF = {"SegFormer_b2": "nvidia/mit-b2"}

class SegNet(nn.Module):
    def __init__(self, name, nc):
        super().__init__()
        if name in SMP_SPECS:
            arch, enc = SMP_SPECS[name]
            self.net = arch(encoder_name=enc, encoder_weights=None, in_channels=3, classes=nc); self.kind = "smp"
        else:
            self.net = SegformerForSemanticSegmentation.from_pretrained(
                SEGFORMER_HF[name], num_labels=nc, ignore_mismatched_sizes=True); self.kind = "hf"
    def forward(self, x):
        if self.kind == "smp": return self.net(x)
        lg = self.net(pixel_values=x).logits
        return F.interpolate(lg, size=x.shape[-2:], mode="bilinear", align_corners=False)

def evaluate_classifier_on(df, model, img_size):
    tf = T.Compose([T.Resize((img_size, img_size)), T.ToTensor(), T.Normalize(MEAN, STD)])
    model.eval(); y_true, y_pred = [], []
    with torch.no_grad():
        for r in df.itertuples():
            x = tf(Image.open(r.filepath).convert("RGB")).unsqueeze(0).to(device)
            p = int(model(x).argmax(1))
            y_true.append(label2idx[r.label]); y_pred.append(p)
    return (accuracy_score(y_true, y_pred), f1_score(y_true, y_pred, average="macro", zero_division=0),
            matthews_corrcoef(y_true, y_pred))

def evaluate_segmenter_on(df, model):
    seg_tf = T.Compose([T.Resize((IMG_SIZE_SEG, IMG_SIZE_SEG)), T.ToTensor(), T.Normalize(MEAN, STD)])
    mask_tf = T.Resize((IMG_SIZE_SEG, IMG_SIZE_SEG), interpolation=T.InterpolationMode.NEAREST)
    model.eval(); ious = []; dices = []
    with torch.no_grad():
        for r in df.itertuples():
            x = seg_tf(Image.open(r.filepath).convert("RGB")).unsqueeze(0).to(device)
            gt = np.array(mask_tf(Image.open(r.maskpath))) > 0
            pr = (model(x).argmax(1)[0].cpu().numpy() > 0)
            inter = (gt & pr).sum(); union = (gt | pr).sum()
            ious.append(inter/union if union > 0 else 1.0)
            dices.append(2*inter/(gt.sum()+pr.sum()) if (gt.sum()+pr.sum()) > 0 else 1.0)
    return float(np.mean(ious)), float(np.mean(dices))

def ccc_fn(y_true, y_pred):
    y_true, y_pred = np.asarray(y_true), np.asarray(y_pred)
    mt, mp = y_true.mean(), y_pred.mean(); vt, vp = y_true.var(), y_pred.var()
    covar = np.mean((y_true-mt)*(y_pred-mp))
    return (2*covar)/(vt+vp+(mt-mp)**2)

print("Model definitions ready.")

Model definitions ready.


## Section 1 -- Validation-split model re-ranking across 3 seeds

Repeats Stage 6 Section 2's test-set-vs-validation-set re-ranking check, but now across all
3 seeds for both classifiers and segmenters, so the paper can report whether model selection
is stable across training runs rather than a single-seed snapshot.

In [5]:
# ============ SECTION 1a: CLASSIFIERS -- 3-SEED VALIDATION RE-RANKING ============
val_cls_rows = []
for seed in SEEDS:
    for name in CLS_NAMES_ALL:
        try:
            raw = find_and_load_checkpoint(cls_ckpt_candidates(name, seed))
        except FileNotFoundError as e:
            print(f"  [skip] seed{seed} {name}: {e}"); continue
        try:
            model, img_size = build_backbone(name, NUM_CLASSES)
            model = apply_state(model, raw).to(device).eval()
        except Exception as e:
            print(f"  [SKIP] seed{seed} {name}: checkpoint failed to load -- {e}"); continue
        acc, f1m, mcc = evaluate_classifier_on(val_df, model, img_size)
        val_cls_rows.append({"seed": seed, "model": name, "val_accuracy": acc,
                              "val_f1_macro": f1m, "val_mcc": mcc})
        print(f"  seed{seed} {name}: val_acc={acc:.4f} f1_macro={f1m:.4f} mcc={mcc:.4f}")
        del model; torch.cuda.empty_cache()

val_cls_df = pd.DataFrame(val_cls_rows)
val_cls_df.to_csv(OUTDIR/"classifier_val_reranking_3seed.csv", index=False)

agg_cls = val_cls_df.groupby("model")["val_f1_macro"].agg(["mean","std","count"]).sort_values("mean", ascending=False)
print("\nClassifier ranking, mean +/- std across available seeds (val_f1_macro):")
print(agg_cls.round(4).to_string())
winners = val_cls_df.loc[val_cls_df.groupby("seed")["val_f1_macro"].idxmax()]
print("\nBest model per seed:")
print(winners[["seed","model","val_f1_macro"]].to_string(index=False))
print(f"Same model wins at every available seed: {winners['model'].nunique() == 1}")
agg_cls.to_csv(OUTDIR/"classifier_val_reranking_3seed_SUMMARY.csv")

    [find_and_load] loaded /kaggle/input/datasets/alenmaruf/model-pt-class-seg-zip/model pt class+Seg/CLassification/43seed_classifier_checkpoints/MobileNetV3Large_best.pt
  seed42 MobileNetV3Large: val_acc=0.6749 f1_macro=0.5917 mcc=0.6706
    [find_and_load] loaded /kaggle/input/datasets/alenmaruf/model-pt-class-seg-zip/model pt class+Seg/CLassification/43seed_classifier_checkpoints/EfficientNetB0_best.pt
  seed42 EfficientNetB0: val_acc=0.6395 f1_macro=0.5537 mcc=0.6367
    [find_and_load] loaded /kaggle/input/datasets/alenmaruf/model-pt-class-seg-zip/model pt class+Seg/CLassification/43seed_classifier_checkpoints/ResNet50_best.pt
  seed42 ResNet50: val_acc=0.6749 f1_macro=0.5860 mcc=0.6711
    [find_and_load] loaded /kaggle/input/datasets/alenmaruf/model-pt-class-seg-zip/model pt class+Seg/CLassification/43seed_classifier_checkpoints/EfficientNetV2S_best.pt
  seed42 EfficientNetV2S: val_acc=0.7305 f1_macro=0.6492 mcc=0.7271
    [find_and_load] loaded /kaggle/input/datasets/alenmaru

In [6]:
# ============ SECTION 1b: SEGMENTERS -- 3-SEED VALIDATION RE-RANKING ============
val_seg_rows = []
for seed in SEEDS:
    for name in SEG_NAMES_ALL:
        try:
            raw = find_and_load_checkpoint(seg_ckpt_candidates(name, seed))
        except FileNotFoundError as e:
            print(f"  [skip] seed{seed} {name}: {e}"); continue
        try:
            model = SegNet(name, 2); model = apply_state(model, raw).to(device).eval()
        except Exception as e:
            print(f"  [SKIP] seed{seed} {name}: checkpoint failed to load -- {e}"); continue
        fg_iou, dice = evaluate_segmenter_on(val_df, model)
        val_seg_rows.append({"seed": seed, "model": name, "val_fg_iou_macro": fg_iou, "val_dice_macro": dice})
        print(f"  seed{seed} {name}: val_fg_iou={fg_iou:.4f} val_dice={dice:.4f}")
        del model; torch.cuda.empty_cache()

val_seg_df = pd.DataFrame(val_seg_rows)
val_seg_df.to_csv(OUTDIR/"segmenter_val_reranking_3seed.csv", index=False)

agg_seg = val_seg_df.groupby("model")["val_fg_iou_macro"].agg(["mean","std","count"]).sort_values("mean", ascending=False)
print("\nSegmenter ranking, mean +/- std across available seeds (val_fg_iou_macro):")
print(agg_seg.round(4).to_string())
winners_seg = val_seg_df.loc[val_seg_df.groupby("seed")["val_fg_iou_macro"].idxmax()]
print("\nBest model per seed:")
print(winners_seg[["seed","model","val_fg_iou_macro"]].to_string(index=False))
print(f"Same model wins at every available seed: {winners_seg['model'].nunique() == 1}")
agg_seg.to_csv(OUTDIR/"segmenter_val_reranking_3seed_SUMMARY.csv")

    [find_and_load] loaded /kaggle/input/datasets/alenmaruf/model-pt-class-seg-zip/model pt class+Seg/Segmentation/43seed_Seegmentation_checkpoints/UNet_resnet34.pt
  seed42 UNet_resnet34: val_fg_iou=0.5439 val_dice=0.6617
    [find_and_load] loaded /kaggle/input/datasets/alenmaruf/model-pt-class-seg-zip/model pt class+Seg/Segmentation/43seed_Seegmentation_checkpoints/UNetPP_resnet50.pt
  seed42 UNetPP_resnet50: val_fg_iou=0.5258 val_dice=0.6435
    [find_and_load] loaded /kaggle/input/datasets/alenmaruf/model-pt-class-seg-zip/model pt class+Seg/Segmentation/43seed_Seegmentation_checkpoints/FPN_resnet50.pt
  seed42 FPN_resnet50: val_fg_iou=0.5445 val_dice=0.6634
    [find_and_load] loaded /kaggle/input/datasets/alenmaruf/model-pt-class-seg-zip/model pt class+Seg/Segmentation/43seed_Seegmentation_checkpoints/DeepLabV3Plus_resnet50.pt
  seed42 DeepLabV3Plus_resnet50: val_fg_iou=0.5255 val_dice=0.6453
    [find_and_load] loaded /kaggle/input/datasets/alenmaruf/model-pt-class-seg-zip/model

    [find_and_load] loaded /kaggle/input/datasets/alenmaruf/model-pt-class-seg-zip/model pt class+Seg/Segmentation/43seed_Seegmentation_checkpoints/SegFormer_b2.pt


config.json:   0%|          | 0.00/70.0k [00:00<?, ?B/s]

[transformers] You passed `num_labels=2` which is incompatible to the `id2label` map of length `1000`.


pytorch_model.bin: reconstructing file:   0%|          |  0.00B / 99.0MB            

...

    [apply_state] missing=372 unexpected=372
    [apply_state] sample checkpoint keys: ['net.segformer.encoder.patch_embeddings.0.proj.weight', 'net.segformer.encoder.patch_embeddings.0.proj.bias', 'net.segformer.encoder.patch_embeddings.0.layer_norm.weight']
    [apply_state] sample expected keys  : ['net.segformer.stages.0.patch_embeddings.proj.weight', 'net.segformer.stages.0.patch_embeddings.proj.bias', 'net.segformer.stages.0.patch_embeddings.layer_norm.weight']
  [SKIP] seed123 SegFormer_b2: checkpoint failed to load -- >=50% of keys unmatched (missing=372, unexpected=372) -- checkpoint did not load; refusing to report numbers from random weights
    [find_and_load] found nested extracted checkpoint folder /kaggle/input/datasets/alenmaruf/model-pt-class-seg-zip/model pt class+Seg/Segmentation/7seed_Seegmentation_checkpoints/UNet_resnet34_seed7/UNet_resnet34_seed7; re-zipping in memory
  seed7 UNet_resnet34: val_fg_iou=0.5406 val_dice=0.6600
    [find_and_load] found nested extrac

[transformers] You passed `num_labels=2` which is incompatible to the `id2label` map of length `1000`.


Loading weights:   0%|          | 0/364 [00:00<?, ?it/s]

[transformers] SegformerForSemanticSegmentation LOAD REPORT from: nvidia/mit-b2
Key                                                     | Status     | 
--------------------------------------------------------+------------+-
classifier.weight                                       | UNEXPECTED | 
classifier.bias                                         | UNEXPECTED | 
decode_head.linear_projections.{0, 1, 2, 3}.proj.weight | MISSING    | 
decode_head.classifier.bias                             | MISSING    | 
decode_head.linear_fuse.weight                          | MISSING    | 
decode_head.classifier.weight                           | MISSING    | 
decode_head.batch_norm.running_var                      | MISSING    | 
decode_head.linear_projections.{0, 1, 2, 3}.proj.bias   | MISSING    | 
decode_head.batch_norm.running_mean                     | MISSING    | 
decode_head.batch_norm.num_batches_tracked              | MISSING    | 
decode_head.batch_norm.weight                           

    [apply_state] missing=372 unexpected=372
    [apply_state] sample checkpoint keys: ['net.segformer.encoder.patch_embeddings.0.proj.weight', 'net.segformer.encoder.patch_embeddings.0.proj.bias', 'net.segformer.encoder.patch_embeddings.0.layer_norm.weight']
    [apply_state] sample expected keys  : ['net.segformer.stages.0.patch_embeddings.proj.weight', 'net.segformer.stages.0.patch_embeddings.proj.bias', 'net.segformer.stages.0.patch_embeddings.layer_norm.weight']
  [SKIP] seed7 SegFormer_b2: checkpoint failed to load -- >=50% of keys unmatched (missing=372, unexpected=372) -- checkpoint did not load; refusing to report numbers from random weights

Segmenter ranking, mean +/- std across available seeds (val_fg_iou_macro):
                                 mean     std  count
model                                               
DeepLabV3Plus_efficientnet-b3  0.5693  0.0198      3
FPN_resnet50                   0.5441  0.0023      3
UNet_resnet34                  0.5418  0.0018      3


## Section 2 -- Validation-split confidence-flag check across 3 seeds

Reruns Stage 5's deployed-pipeline flag validation (classification correctness and severity
error, stratified by the Grad-CAM-derived HIGH/LOW confidence flag) on the **validation**
split, once per seed, using that seed's own ConvNeXtV2-Tiny + DeepLabV3+/EfficientNet-B3
checkpoints.

In [7]:
# ============ SECTION 2: VALIDATION-SPLIT CONFIDENCE-FLAG CHECK, 3 SEEDS ============
cls_tf = T.Compose([T.Resize((IMG_SIZE_CLS, IMG_SIZE_CLS)), T.ToTensor(), T.Normalize(MEAN, STD)])
seg_tf = T.Compose([T.Resize((IMG_SIZE_SEG, IMG_SIZE_SEG)), T.ToTensor(), T.Normalize(MEAN, STD)])

flag_check_rows = []
flag_summary_rows = []
for seed in SEEDS:
    try:
        clf_raw = find_and_load_checkpoint(cls_ckpt_candidates("ConvNeXtV2Tiny", seed))
        seg_raw = find_and_load_checkpoint(seg_ckpt_candidates("DeepLabV3Plus_efficientnet-b3", seed))
    except FileNotFoundError as e:
        print(f"  [skip seed {seed}] {e}"); continue
    try:
        clf = build_classifier(NUM_CLASSES); clf = apply_state(clf, clf_raw).to(device).eval()
        seg = SegNet("DeepLabV3Plus_efficientnet-b3", 2); seg = apply_state(seg, seg_raw).to(device).eval()
    except Exception as e:
        print(f"  [SKIP seed {seed}] checkpoint failed to load -- {e}"); continue
    cam_extractor = GradCAM(model=clf, target_layers=[clf.stages[-1]])

    for _, r in val_df.iterrows():
        img = Image.open(r["filepath"]).convert("RGB")
        x_cls = cls_tf(img).unsqueeze(0).to(device)
        with torch.no_grad():
            probs = torch.softmax(clf(x_cls), 1)[0]
        pred_idx = int(probs.argmax()); pred_class = CLASS_NAMES[pred_idx]
        correct = bool(pred_class == r["Disease"])

        x_seg = seg_tf(img).unsqueeze(0).to(device)
        with torch.no_grad():
            pred_mask = seg(x_seg).argmax(1)[0].cpu().numpy().astype(np.uint8)
        severity_pred = float((pred_mask > 0).mean())
        severity_gt = float(r["Mask ratio"]) if "Mask ratio" in r and pd.notna(r["Mask ratio"]) else np.nan
        severity_abs_err = abs(severity_pred - severity_gt) if pd.notna(severity_gt) else np.nan

        cam = cam_extractor(input_tensor=x_cls, targets=None)[0]
        cam_rs = np.array(Image.fromarray((cam*255).astype(np.uint8)).resize((IMG_SIZE_SEG, IMG_SIZE_SEG))) / 255.0
        tau = max(severity_pred, 1e-3)
        thresh = np.quantile(cam_rs, 1 - tau)
        cam_bin = (cam_rs >= thresh)
        inter = np.logical_and(cam_bin, pred_mask > 0).sum(); union = np.logical_or(cam_bin, pred_mask > 0).sum()
        cam_pred_iou = float(inter/union) if union > 0 else 0.0
        peak_yx = np.unravel_index(np.argmax(cam_rs), cam_rs.shape)
        pointing_hit = bool(pred_mask[peak_yx] > 0)
        confidence_flag = "HIGH" if pointing_hit else "LOW"

        flag_check_rows.append({"seed": seed, "image": r["Name"], "correct": correct,
                                 "severity_abs_error": severity_abs_err, "cam_pred_iou": cam_pred_iou,
                                 "pointing_hit": pointing_hit, "confidence_flag": confidence_flag})

    del clf, seg, cam_extractor; torch.cuda.empty_cache()
    print(f"  seed {seed}: validation-split flag check done ({len(val_df)} images)")

flag_df = pd.DataFrame(flag_check_rows)
flag_df.to_csv(OUTDIR/"confidence_flag_val_3seed_per_image.csv", index=False)

for seed, grp in flag_df.groupby("seed"):
    ct = pd.crosstab(grp.confidence_flag, grp.correct)
    odds, p_fisher = (np.nan, np.nan)
    if ct.shape == (2,2):
        odds, p_fisher = fisher_exact(ct.values)
    high_err = grp.loc[grp.confidence_flag=="HIGH","severity_abs_error"].dropna()
    low_err  = grp.loc[grp.confidence_flag=="LOW", "severity_abs_error"].dropna()
    u_stat, p_mwu = (np.nan, np.nan)
    if len(high_err) > 0 and len(low_err) > 0:
        u_stat, p_mwu = mannwhitneyu(high_err, low_err, alternative="two-sided")
    acc_high = grp.loc[grp.confidence_flag=="HIGH","correct"].mean()
    acc_low  = grp.loc[grp.confidence_flag=="LOW", "correct"].mean()
    flag_summary_rows.append({"seed": seed, "n": len(grp),
                               "high_conf_rate": float((grp.confidence_flag=="HIGH").mean()),
                               "accuracy_high": float(acc_high), "accuracy_low": float(acc_low),
                               "fisher_odds": float(odds) if pd.notna(odds) else None,
                               "fisher_p": float(p_fisher) if pd.notna(p_fisher) else None,
                               "mwu_p_severity_error": float(p_mwu) if pd.notna(p_mwu) else None})
    print(f"  seed {seed}: HIGH acc={acc_high:.4f} LOW acc={acc_low:.4f} "
          f"fisher_p={p_fisher:.4f} mwu_p={p_mwu:.4f}")

flag_summary = pd.DataFrame(flag_summary_rows)
flag_summary.to_csv(OUTDIR/"confidence_flag_val_3seed_SUMMARY.csv", index=False)
print("\nSaved: confidence_flag_val_3seed_per_image.csv and _SUMMARY.csv")
print("Reading guide: if accuracy_high > accuracy_low and fisher_p stays small at every seed,")
print("the confidence flag's validity does not depend on one particular training run.")

  seed 42: validation-split flag check done (846 images)
  seed 123: validation-split flag check done (846 images)
  seed 7: validation-split flag check done (846 images)
  seed 7: HIGH acc=0.7718 LOW acc=0.7253 fisher_p=0.1272 mwu_p=0.7134
  seed 42: HIGH acc=0.7171 LOW acc=0.6333 fisher_p=0.0098 mwu_p=0.3574
  seed 123: HIGH acc=0.7677 LOW acc=0.7179 fisher_p=0.1087 mwu_p=0.6872

Saved: confidence_flag_val_3seed_per_image.csv and _SUMMARY.csv
Reading guide: if accuracy_high > accuracy_low and fisher_p stays small at every seed,
the confidence flag's validity does not depend on one particular training run.


## Section 3 -- Consistent 3-seed segmentation / severity / attribution recompute

For all 6 segmenters at all 3 seeds: test-set fg-IoU/Dice, severity CCC/MAE against
`Mask ratio`, and CAM-GT faithfulness on the same 300-image stratified subset used
throughout the paper (`NAMES_300`, identical list to Notebook A / Stage 4 / Stage 6), so the
paper can report mean +/- std across seeds instead of single-seed point estimates for every
one of these three quantities.

In [8]:
# ============ SECTION 3: 3-SEED SEGMENTATION / SEVERITY / ATTRIBUTION RECOMPUTE ============
NAMES_300 = ['apple_black_rot_google_0037.jpg', 'apple_black_rot_google_0077.jpg', 'apple_mosaic_virus_57.jpg', 'apple_mosaic_virus_google_0074.jpg', 'apple_rust_147.jpg', 'apple_rust_google_0007.jpg', 'apple_rust_google_0011.jpg', 'apple_scab_50.jpg', 'apple_scab_85.jpg', 'apple_scab_google_0018.jpg', 'apple_scab_google_0031.jpg', 'apple_scab_google_0153.jpg', 'apple_scab_google_0244.jpg', 'banana_anthracnose_Baidu_0059.jpg', 'banana_anthracnose_Bing_0006.jpg', 'banana_anthracnose_Bing_0022.jpg', 'banana_anthracnose_Bing_0032.jpg', 'banana_black_leaf_streak_Bing_0124.jpg', 'banana_black_leaf_streak_banana black sigatoka (26).jpg', 'banana_bunchy_top_Bing_0152.jpg', 'banana_bunchy_top_Google_0037.jpg', 'banana_cigar_end_rot_Bing_0007.jpg', 'banana_cigar_end_rot_Google_0101.jpg', 'banana_cordana_leaf_spot_Bing_0007.jpg', 'banana_cordana_leaf_spot_Bing_0037.jpg', 'banana_panama_disease_126.jpg', 'banana_panama_disease_google_0064.jpg', 'basil_downy_mildew_google_0031.jpg', 'basil_downy_mildew_google_0091.jpg', 'bean_halo_blight_11.jpg', 'bean_halo_blight_2.jpg', 'bean_mosaic_virus_24.jpg', 'bean_mosaic_virus_google_0098.jpg', 'bean_rust_google_0045.jpg', 'bean_rust_google_0068.jpg', 'bell_pepper_bacterial_spot_Bing_0002.jpg', 'bell_pepper_bacterial_spot_Bing_0028.jpg', 'bell_pepper_blossom_end_rot_Bing_0002.jpg', 'bell_pepper_blossom_end_rot_Bing_0004.jpg', 'bell_pepper_blossom_end_rot_Bing_0077.jpg', 'bell_pepper_frogeye_leaf_spot_Bing_0008.jpg', 'bell_pepper_frogeye_leaf_spot_Google_0049.jpg', 'bell_pepper_powdery_mildew_Bing_0035.jpg', 'bell_pepper_powdery_mildew_Google_0118.jpg', 'blueberry_anthracnose_Bing_0006.jpg', 'blueberry_anthracnose_Bing_0145.jpg', 'blueberry_botrytis_blight_Bing_0015.jpg', 'blueberry_botrytis_blight_Bing_0021.jpg', 'blueberry_mummy_berry_Bing_0024.jpg', 'blueberry_mummy_berry_Google_0038.jpg', 'blueberry_rust_google_0004.jpg', 'blueberry_rust_google_0053.jpg', 'blueberry_scorch_Bing_0011.jpg', 'blueberry_scorch_Bing_0031.jpg', 'blueberry_scorch_Bing_0062.jpg', 'broccoli_alternaria_leaf_spot_Bing_0000.jpg', 'broccoli_alternaria_leaf_spot_Bing_0054.jpg', 'broccoli_downy_mildew_30.jpg', 'broccoli_downy_mildew_40.jpg', 'broccoli_ring_spot_Bing_0002.jpg', 'broccoli_ring_spot_Bing_0018.jpg', 'cabbage_alternaria_leaf_spot_3.jpg', 'cabbage_alternaria_leaf_spot_google_0206.jpg', 'cabbage_black_rot_Bing_0002.jpg', 'cabbage_black_rot_Bing_0018.jpg', 'cabbage_black_rot_Bing_0039.jpg', 'cabbage_black_rot_Bing_0080.jpg', 'cabbage_black_rot_Bing_0114.jpg', 'cabbage_downy_mildew_Bing_0012.jpg', 'cabbage_downy_mildew_Bing_0124.jpg', 'carrot_alternaria_leaf_blight_Bing_0046.jpg', 'carrot_alternaria_leaf_blight_Bing_0071.jpg', 'carrot_cavity_spot_google_0000.jpg', 'carrot_cavity_spot_google_0161.jpg', 'carrot_cercospora_leaf_blight_Bing_0009.jpg', 'carrot_cercospora_leaf_blight_Bing_0212.jpg', 'cauliflower_alternaria_leaf_spot_google_0003.jpg', 'cauliflower_alternaria_leaf_spot_google_0061.jpg', 'cauliflower_bacterial_soft_rot_Bing_0021.jpg', 'cauliflower_bacterial_soft_rot_Bing_0205.jpg', 'celery_anthracnose_google_0007.jpg', 'celery_anthracnose_google_0035.jpg', 'celery_early_blight_52.jpg', 'celery_early_blight_google_0001.jpg', 'celery_early_blight_google_0009.jpg', 'cherry_leaf_spot_google_0180.jpg', 'cherry_leaf_spot_google_0216.jpg', 'cherry_powdery_mildew_107.jpg', 'cherry_powdery_mildew_2.jpg', 'citrus_canker_13.jpg', 'citrus_canker_147.jpg', 'citrus_canker_163.jpg', 'citrus_canker_247.jpg', 'citrus_canker_363.jpg', 'citrus_canker_76.jpg', 'citrus_canker_89.jpg', 'citrus_canker_google_0071.jpg', 'citrus_greening_disease_29.jpg', 'citrus_greening_disease_48.jpg', 'citrus_greening_disease_google_0120.jpg', 'citrus_greening_disease_google_0251.jpg', 'coffee_berry_blotch_Bing_0027.jpg', 'coffee_berry_blotch_Bing_0114.jpg', 'coffee_berry_blotch_Bing_0120.jpg', 'coffee_brown_eye_spot_Bing_0036.jpg', 'coffee_brown_eye_spot_Google_0003.jpg', 'coffee_leaf_rust_15.jpg', 'coffee_leaf_rust_google_0016.jpg', 'coffee_leaf_rust_google_0135.jpg', 'corn_gray_leaf_spot_31.jpg', 'corn_gray_leaf_spot_35.jpg', 'corn_gray_leaf_spot_google_0074.jpg', 'corn_northern_leaf_blight_101.jpg', 'corn_northern_leaf_blight_14.jpg', 'corn_northern_leaf_blight_81.jpg', 'corn_rust_80.jpg', 'corn_rust_google_0078.jpg', 'corn_smut_google_0013.jpg', 'corn_smut_google_0063.jpg', 'corn_smut_google_0079.jpg', 'cucumber_angular_leaf_spot_1.jpg', 'cucumber_angular_leaf_spot_156.jpg', 'cucumber_angular_leaf_spot_328.jpg', 'cucumber_angular_leaf_spot_41.jpg', 'cucumber_angular_leaf_spot_google_0043.jpg', 'cucumber_bacterial_wilt_109.jpg', 'cucumber_bacterial_wilt_313.jpg', 'cucumber_bacterial_wilt_50.jpg', 'cucumber_powdery_mildew_58.jpg', 'cucumber_powdery_mildew_70.jpg', 'cucumber_powdery_mildew_77.jpg', 'cucumber_powdery_mildew_google_0159.jpg', 'eggplant_cercospora_leaf_spot_12.jpg', 'eggplant_cercospora_leaf_spot_33.jpg', 'eggplant_cercospora_leaf_spot_40.jpg', 'eggplant_phomopsis_fruit_rot_Bing_0016.jpg', 'eggplant_phomopsis_fruit_rot_Bing_0022.jpg', 'eggplant_phomopsis_fruit_rot_Bing_0261.jpg', 'eggplant_phytophthora_blight_Bing_0000.jpg', 'eggplant_phytophthora_blight_Bing_0002.jpg', 'garlic_leaf_blight_140.jpg', 'garlic_leaf_blight_46.jpg', 'garlic_leaf_blight_google_0004.jpg', 'garlic_leaf_blight_google_0210.jpg', 'garlic_rust_159.jpg', 'garlic_rust_18.jpg', 'ginger_leaf_spot_11.jpg', 'ginger_leaf_spot_google_0001.jpg', 'ginger_sheath_blight_53.jpg', 'ginger_sheath_blight_89.jpg', 'grape_black_rot_12.jpg', 'grape_black_rot_216.jpg', 'grape_black_rot_google_0013.jpg', 'grape_downy_mildew_251.jpg', 'grape_downy_mildew_272.jpg', 'grape_downy_mildew_google_0032.jpg', 'grape_downy_mildew_google_0052.jpg', 'grape_leaf_spot_101.jpg', 'grape_leaf_spot_168.jpg', 'grapevine_leafroll_disease_google_0001.jpg', 'grapevine_leafroll_disease_google_0181.jpg', 'lettuce_downy_mildew_google_0015.jpg', 'lettuce_downy_mildew_google_0027.jpg', 'lettuce_mosaic_virus_16.jpg', 'lettuce_mosaic_virus_24.jpg', 'maple_tar_spot_google_0005.jpg', 'maple_tar_spot_google_0089.jpg', 'peach_anthracnose_Bing_0002.jpg', 'peach_anthracnose_Bing_0013.jpg', 'peach_brown_rot_Bing_0000.jpg', 'peach_brown_rot_Bing_0092.jpg', 'peach_brown_rot_Bing_0114.jpg', 'peach_leaf_curl_34.jpg', 'peach_leaf_curl_google_0020.jpg', 'peach_leaf_curl_google_0035.jpg', 'peach_leaf_curl_google_0074.jpg', 'peach_rust_Bing_0001.jpg', 'peach_scab_Bing_0007.jpg', 'peach_scab_Bing_0083.jpg', 'plum_bacterial_spot_Bing_0013.jpg', 'plum_bacterial_spot_Bing_0082.jpg', 'plum_brown_rot_Bing_0001.jpg', 'plum_brown_rot_Bing_0097.jpg', 'plum_pocket_disease_google_0028.jpg', 'plum_pocket_disease_google_0032.jpg', 'plum_pocket_disease_google_0117.jpg', 'plum_pox_virus_Bing_0052.jpg', 'plum_pox_virus_Bing_0054.jpg', 'plum_pox_virus_Google_0040.jpg', 'plum_rust_Bing_0024.jpg', 'plum_rust_Google_0036.jpg', 'potato_early_blight_6.jpg', 'potato_early_blight_google_0095.jpg', 'potato_late_blight_11.jpg', 'potato_late_blight_36.jpg', 'potato_late_blight_96.jpg', 'raspberry_fire_blight_Bing_0018.jpg', 'raspberry_fire_blight_Google_0005.jpg', 'raspberry_gray_mold_Bing_0000.jpg', 'raspberry_gray_mold_Bing_0001.jpg', 'raspberry_leaf_spot_Bing_0073.jpg', 'raspberry_yellow_rust_Bing_0010.jpg', 'raspberry_yellow_rust_Bing_0183.jpg', 'rice_blast_151.jpg', 'rice_blast_87.jpg', 'rice_sheath_blight_148.jpg', 'rice_sheath_blight_google_0027.jpg', 'soybean_bacterial_blight_Bing_0057.jpg', 'soybean_bacterial_blight_Bing_0105.jpg', 'soybean_bacterial_blight_Bing_0172.jpg', 'soybean_brown_spot_Baidu_0010.jpg', 'soybean_brown_spot_Bing_0353.jpg', 'soybean_downy_mildew_Bing_0078.jpg', 'soybean_downy_mildew_Bing_0143.jpg', 'soybean_frog_eye_leaf_spot_Bing_0141.jpg', 'soybean_frog_eye_leaf_spot_Google_0057.jpg', 'soybean_frog_eye_leaf_spot_Google_0221.jpg', 'soybean_mosaic_Baidu_0059.jpg', 'soybean_mosaic_Baidu_0089.jpg', 'soybean_rust_Bing_0022.jpg', 'soybean_rust_Bing_0055.jpg', 'soybean_rust_Bing_0074.jpg', 'soybean_rust_Bing_0075.jpg', 'soybean_rust_Bing_0096.jpg', 'squash_powdery_mildew_129.jpg', 'squash_powdery_mildew_23.jpg', 'squash_powdery_mildew_6.jpg', 'squash_powdery_mildew_google_0105.jpg', 'strawberry_anthracnose_5.jpg', 'strawberry_anthracnose_google_0001.jpg', 'strawberry_leaf_scorch_google_0037.jpg', 'strawberry_leaf_scorch_google_0133.jpg', 'tobacco_blue_mold_Bing_0024.jpg', 'tobacco_blue_mold_Bing_0096.jpg', 'tobacco_brown_spot_Bing_0001.jpg', 'tobacco_brown_spot_Bing_0047.jpg', 'tobacco_brown_spot_Bing_0055.jpg', 'tobacco_frogeye_leaf_spot_Bing_0002.jpg', 'tobacco_frogeye_leaf_spot_Google_0052.jpg', 'tobacco_mosaic_virus_117.jpg', 'tobacco_mosaic_virus_4.jpg', 'tomato_bacterial_leaf_spot_120.jpg', 'tomato_bacterial_leaf_spot_14.jpg', 'tomato_early_blight_96.jpg', 'tomato_early_blight_google_0059.jpg', 'tomato_early_blight_google_0177.jpg', 'tomato_late_blight_10.jpg', 'tomato_late_blight_182.jpg', 'tomato_late_blight_205.jpg', 'tomato_leaf_mold_10.jpg', 'tomato_leaf_mold_148.jpg', 'tomato_mosaic_virus_15.jpg', 'tomato_mosaic_virus_google_0049.jpg', 'tomato_septoria_leaf_spot_10.jpg', 'tomato_septoria_leaf_spot_18.jpg', 'tomato_septoria_leaf_spot_243.jpg', 'tomato_septoria_leaf_spot_80.jpg', 'tomato_septoria_leaf_spot_google_0027.jpg', 'tomato_septoria_leaf_spot_google_0070.jpg', 'tomato_septoria_leaf_spot_google_0128.jpg', 'tomato_yellow_leaf_curl_virus_15.jpg', 'tomato_yellow_leaf_curl_virus_17.jpg', 'tomato_yellow_leaf_curl_virus_40.jpg', 'tomato_yellow_leaf_curl_virus_99.jpg', 'wheat_bacterial_leaf_streak_(black_chaff)_Baidu_0031.jpg', 'wheat_bacterial_leaf_streak_(black_chaff)_Baidu_0235.jpg', 'wheat_head_scab_Baidu_0481.jpg', 'wheat_head_scab_Bing_0040.jpg', 'wheat_head_scab_Bing_0043.jpg', 'wheat_head_scab_Bing_0288.jpg', 'wheat_leaf_rust_Google_0040.jpg', 'wheat_leaf_rust_Google_0058.jpg', 'wheat_leaf_rust_Google_0328.jpg', 'wheat_loose_smut_Bing_0109.jpg', 'wheat_loose_smut_Bing_0283.jpg', 'wheat_loose_smut_Google_0081.jpg', 'wheat_powdery_mildew_Baidu_0181.jpg', 'wheat_powdery_mildew_Bing_0014.jpg', 'wheat_powdery_mildew_Google_0014.jpg', 'wheat_septoria_blotch_Google_0115.jpg', 'wheat_septoria_blotch_google_septoria (18).jpg', 'wheat_stem_rust_Baidu_0001.jpg', 'wheat_stem_rust_Bing_0217.jpg', 'wheat_stem_rust_Google_0274.jpg', 'wheat_stripe_rust_Baidu_0143.jpg', 'wheat_stripe_rust_Bing_0007.jpg', 'wheat_stripe_rust_Bing_0257.jpg', 'wheat_stripe_rust_Google_0029.jpg', 'zucchini_bacterial_wilt_Bing_0105.jpg', 'zucchini_bacterial_wilt_Bing_0188.jpg', 'zucchini_bacterial_wilt_Bing_0260.jpg', 'zucchini_bacterial_wilt_Google_0060.jpg', 'zucchini_downy_mildew_Bing_0035.jpg', 'zucchini_downy_mildew_Bing_0259.jpg', 'zucchini_powdery_mildew_Bing_0109.jpg', 'zucchini_powdery_mildew_Bing_0155.jpg', 'zucchini_powdery_mildew_Bing_0233.jpg', 'zucchini_yellow_mosaic_virus_11.jpg', 'zucchini_yellow_mosaic_virus_google_0038.jpg', 'zucchini_yellow_mosaic_virus_google_0067.jpg']
sub_df = test_df[test_df["Name"].isin(NAMES_300)].reset_index(drop=True)
assert len(sub_df) == len(NAMES_300)

seg_eval_tf = T.Compose([T.Resize((IMG_SIZE_SEG, IMG_SIZE_SEG)), T.ToTensor(), T.Normalize(MEAN, STD)])
mask_tf = T.Resize((IMG_SIZE_SEG, IMG_SIZE_SEG), interpolation=T.InterpolationMode.NEAREST)
DEL_STEPS = 10

class _GradCAMHook:
    def __init__(self, module):
        self.activations = None; self.gradients = None
        self.fh = module.register_forward_hook(self._fwd)
        self.bh = module.register_full_backward_hook(self._bwd)
    def _fwd(self, m, inp, out): self.activations = out
    def _bwd(self, m, gi, go):  self.gradients  = go[0]
    def remove(self): self.fh.remove(); self.bh.remove()

def gradcam_map(model, x, nc):
    hook = _GradCAMHook(model.net.decoder)
    model.zero_grad(set_to_none=True)
    logits = model(x)
    score = logits[:,1] if nc==2 else logits[:,1:].sum(1)
    score.sum().backward()
    acts  = hook.activations[0].detach(); grads = hook.gradients[0].detach(); hook.remove()
    w = grads.mean(dim=(1,2))
    cam = torch.relu((w[:,None,None]*acts).sum(0)); cam = cam/(cam.max()+1e-8)
    cam = F.interpolate(cam[None,None], size=x.shape[-2:], mode="bilinear", align_corners=False)[0,0]
    return cam.detach().cpu().numpy()

def segformer_attention_map(model, x):
    with torch.no_grad():
        out = model.net.segformer(pixel_values=x, output_attentions=True) \
              if hasattr(model.net, "segformer") else model.net(pixel_values=x, output_attentions=True)
    attns = out.attentions
    if not attns:
        raise RuntimeError("segformer_attention_map: model returned no attentions (output_attentions=True "
                            "had no effect) -- likely a failed/garbled checkpoint load or a transformers "
                            "version mismatch; check the apply_state warning printed for this model")
    stages=[]; cur=[]; cur_n=None
    for a in attns:
        n=a.shape[-2]
        if cur_n is None or n==cur_n: cur.append(a); cur_n=n
        else: stages.append(cur); cur=[a]; cur_n=n
    stages.append(cur)
    rollout=None
    for a in stages[-1]:
        A = a[0].mean(0); A = A/A.sum(-1,keepdim=True).clamp_min(1e-8)
        A_hat = 0.5*A + 0.5*torch.eye(A.shape[0], device=A.device)
        rollout = A_hat if rollout is None else A_hat@rollout
    imp = rollout.mean(0); side = int(round(imp.shape[0]**0.5))
    grid = imp[:side*side].reshape(side,side).float(); grid = grid/(grid.max()+1e-8)
    cam  = F.interpolate(grid[None,None], size=x.shape[-2:], mode="bilinear", align_corners=False)[0,0]
    return cam.detach().cpu().numpy()

def explain(model, x, nc, is_transformer):
    return segformer_attention_map(model,x) if is_transformer else gradcam_map(model,x,nc)

def cam_gt_metrics(cam, gt_mask):
    if gt_mask.sum() == 0: return {"iou": np.nan, "pointing_hit": np.nan}
    thresh = np.quantile(cam, 1 - gt_mask.mean())
    cam_bin = cam >= thresh
    inter = (cam_bin & gt_mask).sum(); union = (cam_bin | gt_mask).sum()
    iou = float(inter/union) if union > 0 else np.nan
    py, px = np.unravel_index(np.argmax(cam), cam.shape)
    return {"iou": iou, "pointing_hit": bool(gt_mask[py, px])}

def deletion_auc(model, x, cam, nc, steps=DEL_STEPS):
    H, W = x.shape[-2:]; order = np.argsort(-cam.ravel()); npix = order.size
    baseline = x.mean(dim=(2,3), keepdim=True); fracs = np.linspace(0, 1, steps+1); probs = []
    with torch.no_grad():
        for f in fracs:
            k = int(f*npix); xin = x.clone()
            if k > 0:
                mask = torch.zeros(npix, dtype=torch.bool, device=x.device)
                mask[order[:k]] = True; mask = mask.reshape(H, W)
                for c in range(xin.shape[1]): xin[0,c][mask] = baseline[0,c,0,0]
            logits = model(xin)
            p = torch.softmax(logits,1)[0,1].mean().item() if nc==2 \
                else torch.softmax(logits,1)[0,1:].sum(0).mean().item()
            probs.append(p)
    return float(np.trapz(probs, fracs))

recompute_rows = []
for seed in SEEDS:
    for name in SEG_NAMES_ALL:
        try:
            raw = find_and_load_checkpoint(seg_ckpt_candidates(name, seed))
        except FileNotFoundError as e:
            print(f"  [skip] seed{seed} {name}: {e}"); continue
        try:
            model = SegNet(name, 2); model = apply_state(model, raw).to(device).eval()
            is_tf = (name == "SegFormer_b2")

            # (a) test-set fg-IoU / Dice
            fg_iou, dice = evaluate_segmenter_on(test_df, model)

            # (b) severity CCC / MAE against Mask ratio (full test set)
            sev_pred=[]; sev_gt=[]
            with torch.no_grad():
                for _, r in test_df.iterrows():
                    if "Mask ratio" not in r or pd.isna(r["Mask ratio"]): continue
                    x = seg_eval_tf(Image.open(r["filepath"]).convert("RGB")).unsqueeze(0).to(device)
                    pr = (model(x).argmax(1)[0].cpu().numpy() > 0)
                    sev_pred.append(float(pr.mean())); sev_gt.append(float(r["Mask ratio"]))
            severity_ccc = ccc_fn(sev_gt, sev_pred) if len(sev_gt) > 1 else np.nan
            severity_mae = float(np.mean(np.abs(np.array(sev_pred)-np.array(sev_gt)))) if sev_gt else np.nan

            # (c) CAM-GT faithfulness on the 300-image subset
            ious=[]; hits=[]; delaucs=[]
            for r in sub_df.itertuples():
                x = seg_eval_tf(Image.open(r.filepath).convert("RGB")).unsqueeze(0).to(device)
                gt = np.array(mask_tf(Image.open(r.maskpath))) > 0
                cam = explain(model, x, 2, is_tf)
                m = cam_gt_metrics(cam, gt)
                ious.append(m["iou"]); hits.append(m["pointing_hit"])
                delaucs.append(deletion_auc(model, x, cam, 2))

            recompute_rows.append({"seed": seed, "model": name,
                                    "test_fg_iou": fg_iou, "test_dice": dice,
                                    "severity_ccc": severity_ccc, "severity_mae": severity_mae,
                                    "cam_gt_iou_mean": float(np.nanmean(ious)),
                                    "pointing_hit_mean": float(np.nanmean(hits)),
                                    "deletion_auc_mean": float(np.nanmean(delaucs))})
            print(f"  seed{seed} {name}: fg_iou={fg_iou:.4f} severity_ccc={severity_ccc:.4f} "
                  f"cam_gt_iou={np.nanmean(ious):.4f}")
        except Exception as e:
            print(f"  [SKIP] seed{seed} {name}: failed -- {e}")
        finally:
            try: del model
            except NameError: pass
            torch.cuda.empty_cache()

recompute_df = pd.DataFrame(recompute_rows)
recompute_df.to_csv(OUTDIR/"segmenter_3seed_full_recompute.csv", index=False)

agg_cols = ["test_fg_iou","test_dice","severity_ccc","severity_mae",
            "cam_gt_iou_mean","pointing_hit_mean","deletion_auc_mean"]
agg_recompute = recompute_df.groupby("model")[agg_cols].agg(["mean","std"])
agg_recompute.to_csv(OUTDIR/"segmenter_3seed_full_recompute_SUMMARY.csv")
print("\n3-seed mean +/- std per segmenter (full table saved to CSV):")
print(agg_recompute.round(4).to_string())

  seed42 UNet_resnet34: fg_iou=0.5554 severity_ccc=0.8134 cam_gt_iou=0.4335
  seed42 UNetPP_resnet50: fg_iou=0.5352 severity_ccc=0.8015 cam_gt_iou=0.3803
  seed42 FPN_resnet50: fg_iou=0.5552 severity_ccc=0.7982 cam_gt_iou=0.3920
  seed42 DeepLabV3Plus_resnet50: fg_iou=0.5337 severity_ccc=0.7758 cam_gt_iou=0.4930


[transformers] You passed `num_labels=2` which is incompatible to the `id2label` map of length `1000`.


  seed42 DeepLabV3Plus_efficientnet-b3: fg_iou=0.5668 severity_ccc=0.8145 cam_gt_iou=0.4685


Loading weights:   0%|          | 0/364 [00:00<?, ?it/s]

[transformers] SegformerForSemanticSegmentation LOAD REPORT from: nvidia/mit-b2
Key                                                     | Status     | 
--------------------------------------------------------+------------+-
classifier.weight                                       | UNEXPECTED | 
classifier.bias                                         | UNEXPECTED | 
decode_head.linear_projections.{0, 1, 2, 3}.proj.weight | MISSING    | 
decode_head.classifier.bias                             | MISSING    | 
decode_head.linear_fuse.weight                          | MISSING    | 
decode_head.classifier.weight                           | MISSING    | 
decode_head.batch_norm.running_var                      | MISSING    | 
decode_head.linear_projections.{0, 1, 2, 3}.proj.bias   | MISSING    | 
decode_head.batch_norm.running_mean                     | MISSING    | 
decode_head.batch_norm.num_batches_tracked              | MISSING    | 
decode_head.batch_norm.weight                           

...

[transformers] You passed `num_labels=2` which is incompatible to the `id2label` map of length `1000`.


  seed7 DeepLabV3Plus_efficientnet-b3: fg_iou=0.5634 severity_ccc=0.7905 cam_gt_iou=0.5077


Loading weights:   0%|          | 0/364 [00:00<?, ?it/s]

[transformers] SegformerForSemanticSegmentation LOAD REPORT from: nvidia/mit-b2
Key                                                     | Status     | 
--------------------------------------------------------+------------+-
classifier.weight                                       | UNEXPECTED | 
classifier.bias                                         | UNEXPECTED | 
decode_head.linear_projections.{0, 1, 2, 3}.proj.weight | MISSING    | 
decode_head.classifier.bias                             | MISSING    | 
decode_head.linear_fuse.weight                          | MISSING    | 
decode_head.classifier.weight                           | MISSING    | 
decode_head.batch_norm.running_var                      | MISSING    | 
decode_head.linear_projections.{0, 1, 2, 3}.proj.bias   | MISSING    | 
decode_head.batch_norm.running_mean                     | MISSING    | 
decode_head.batch_norm.num_batches_tracked              | MISSING    | 
decode_head.batch_norm.weight                           

    [apply_state] missing=372 unexpected=372
    [apply_state] sample checkpoint keys: ['net.segformer.encoder.patch_embeddings.0.proj.weight', 'net.segformer.encoder.patch_embeddings.0.proj.bias', 'net.segformer.encoder.patch_embeddings.0.layer_norm.weight']
    [apply_state] sample expected keys  : ['net.segformer.stages.0.patch_embeddings.proj.weight', 'net.segformer.stages.0.patch_embeddings.proj.bias', 'net.segformer.stages.0.patch_embeddings.layer_norm.weight']
  [SKIP] seed7 SegFormer_b2: failed -- >=50% of keys unmatched (missing=372, unexpected=372) -- checkpoint did not load; refusing to report numbers from random weights

3-seed mean +/- std per segmenter (full table saved to CSV):
                              test_fg_iou         test_dice         severity_ccc         severity_mae         cam_gt_iou_mean         pointing_hit_mean         deletion_auc_mean        
                                     mean     std      mean     std         mean     std         mean     std   

## Wrap-up -- zip all outputs for download

In [9]:
import zipfile as _zf
zip_path = Path("/kaggle/working/plantseg_r2_notebookB_results.zip")
with _zf.ZipFile(zip_path, "w", _zf.ZIP_DEFLATED) as zf:
    for p in sorted(OUTDIR.glob("*")):
        if p.is_file():
            zf.write(p, p.name)
print(f"Archive: {zip_path.name}  ({zip_path.stat().st_size/1024/1024:.2f} MB)")
print("Files included:")
for p in sorted(OUTDIR.glob("*")):
    print(" ", p.name)
print("\nDownload from the Kaggle Output tab and bring all CSV files back for the response letter.")

Archive: plantseg_r2_notebookB_results.zip  (0.07 MB)
Files included:
  classifier_val_reranking_3seed.csv
  classifier_val_reranking_3seed_SUMMARY.csv
  confidence_flag_val_3seed_SUMMARY.csv
  confidence_flag_val_3seed_per_image.csv
  segmenter_3seed_full_recompute.csv
  segmenter_3seed_full_recompute_SUMMARY.csv
  segmenter_val_reranking_3seed.csv
  segmenter_val_reranking_3seed_SUMMARY.csv

Download from the Kaggle Output tab and bring all CSV files back for the response letter.
